In [1]:
%load_ext autoreload
%autoreload 2

In [3]:
import pandas as pd
import networkx as nx
from src.network_metrics import build_metrics_dataframe, calculate_gini_index, calculate_p_impact_index, calculate_zimmermann_p_impact
from pathlib import Path

# 1. Cargar tu grafo desde la capa Gold
df_nodes = pd.read_parquet("data/03_gold/gold_nodes_2026-05-06.parquet")
df_edges = pd.read_parquet("data/03_gold/gold_edges_2026-05-06.parquet")

# 2. Armar los grafos
G_std = nx.DiGraph()
G_fail = nx.DiGraph()

G_std.add_nodes_from(df_nodes['node_name'])
G_fail.add_nodes_from(df_nodes['node_name'])

G_std.add_edges_from(zip(df_edges['source_node'], df_edges['target_node']))
# G_fail invierte la dirección (target -> source) para propagar el daño
G_fail.add_edges_from(zip(df_edges['target_node'], df_edges['source_node']))

total_nodes = G_std.number_of_nodes()

# 3. Calcular PageRank al vuelo (La métrica estructural)
print("Calculando PageRank de la red...")
pagerank_scores = nx.pagerank(G_std, alpha=0.85)

# 2. Obtener todas las métricas limpias
df_metrics = build_metrics_dataframe(G_fail)

# 3. Calcular métricas de ecosistema
gini = calculate_gini_index(df_metrics['transitive_reverse_deps'].values)
p_impact_5 = calculate_p_impact_index(G_fail, p_threshold_percent=5.0)
p_impact_Z = calculate_zimmermann_p_impact(G_fail, pagerank_scores)

print(f"Desigualdad de dependencias (Gini): {gini:.3f}")
print(f"Paquetes P-Impact (5%): {p_impact_5} paquetes pueden tumbar el ecosistema.")
print(f"Paquetes P-Impact Zimmerman (5%): {p_impact_Z} paquetes pueden tumbar el ecosistema.")

Calculando PageRank de la red...
Desigualdad de dependencias (Gini): 0.890
Paquetes P-Impact (5%): 20 paquetes pueden tumbar el ecosistema.
Paquetes P-Impact Zimmerman (5%): 18 paquetes pueden tumbar el ecosistema.
